# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [264]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [265]:
#Ijuí
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Ijuí\SHP criado\4310207\4310207.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Ijuí\107-0019160-SES-GER-HID-00-EC-TomoI\107-0019160-SES-GER-HID-00-EC-APS\APS_CORSAN.shp')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Ijuí\SHP criado\BACIAS_EC.gpkg')
coluna_nome_bacias = 'Nome'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Ijuí\popdom_bacia_2022-comIgrejas.xlsx'
crs = "EPSG:31981"

## Funções auxiliares

In [266]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das linhas dentro de cada polígono.
    Requer que ambos estejam em um CRS projetado em METROS.
    """
    # Garantir colunas necessárias
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()
    lines = lines_gdf[["geometry"]].copy()

    # Corrigir geometrias inválidas (se necessário)
    if hasattr(polys.geometry, "make_valid"):
        polys["geometry"] = polys.geometry.make_valid()
    else:
        polys["geometry"] = polys.buffer(0)

    # Interseção (recorta as linhas por polígono)
    inter = gpd.overlay(lines, polys, how="intersection")

    # Comprimento em metros
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"].sum()
    
    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [267]:
setores['Densidade'] = setores['v0001']/setores['v0003']
setores['Densidade'].replace([np.inf, -np.inf], np.nan, inplace=True)

C:\Users\gabriel.coimbra\AppData\Local\Temp\ipykernel_14516\325780144.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  setores['Densidade'].replace([np.inf, -np.inf], np.nan, inplace=True)


In [268]:
#bacias_setores[bacias_setores['Nome']=="SB-12A"].replace([np.inf, -np.inf], np.nan, inplace=True)
#bacias_setores[bacias_setores['Nome']=="SB-12A"]

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [269]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [270]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [271]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [272]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

In [273]:
populacao_aps['Densidade'].value_counts()

Densidade
2.242718    1
2.432432    1
2.558140    1
2.507246    1
2.532086    1
           ..
0.781250    1
2.155452    1
1.523132    1
2.649819    1
2.357143    1
Name: count, Length: 117, dtype: int64

##### Cálculo da população com a densidade e n_pontos criado

In [274]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

#pop = populacao_aps['População 2022'].replace([np.inf, -np.inf], np.nan, inplace=True)
pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"Qtd domicílios total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 76271
Qtd domicílios total na APS em 2022 é de 33495


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [275]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [276]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [277]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [278]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Nome,,
SB-01,2600,6261.480628
SB-02,40,102.584014
SB-05,1393,2486.166737
SB-07,1901,3933.143331
SB-08,2144,3947.311264
SB-09,688,1451.053257
SB-0A,473,935.670677
SB-0B,888,1705.401533
SB-0C,354,908.245599


##### Exportar excel final

In [279]:
#bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [280]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps

display(resultado_dompop)

A população total na APS em 2022 é de 76271
Os domicílios totais na APS em 2022 é de 33495


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
Nome,,,,,,
SB-01,2600,6261.480628,0.079309,0.077624,0.083869,0.082095
SB-02,40,102.584014,0.001220,0.001194,0.001374,0.001345
SB-05,1393,2486.166737,0.042492,0.041588,0.033301,0.032597
SB-07,1901,3933.143331,0.057987,0.056755,0.052682,0.051568
SB-08,2144,3947.311264,0.065400,0.064010,0.052872,0.051754
SB-09,688,1451.053257,0.020986,0.020540,0.019436,0.019025
SB-0A,473,935.670677,0.014428,0.014122,0.012533,0.012268
SB-0B,888,1705.401533,0.027087,0.026511,0.022843,0.022360
SB-0C,354,908.245599,0.010798,0.010569,0.012165,0.011908


# Extensão e Área das Bacias

In [281]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Aerolevantamento\Fase 7 - Restituicao Planimetrica Digital (MUB)\Shapes\EixoLogradouro.shp')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Nome")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Nome")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

Nome,SB-01,SB-02,SB-05,SB-07,SB-08,SB-09,SB-0A,SB-0B,SB-0C,SB-10,...,SB-3F,SB-3G,SB-6A,SB-6B,SB-6C,SB-6D,SB-6E,SB-6F,SB-6G,SB-6H
Domicílios,2600.000000,40.000000,1393.000000,1901.000000,2144.000000,688.000000,473.000000,888.000000,354.000000,2343.000000,...,12.000000,28.000000,492.000000,148.000000,516.000000,592.000000,168.000000,9.000000,15.000000,41.000000
Dom % bacias,0.079309,0.001220,0.042492,0.057987,0.065400,0.020986,0.014428,0.027087,0.010798,0.071470,...,0.000366,0.000854,0.015008,0.004515,0.015740,0.018058,0.005125,0.000275,0.000458,0.001251
Dom % APS,0.077624,0.001194,0.041588,0.056755,0.064010,0.020540,0.014122,0.026511,0.010569,0.069951,...,0.000358,0.000836,0.014689,0.004419,0.015405,0.017674,0.005016,0.000269,0.000448,0.001224
População,6261.480628,102.584014,2486.166737,3933.143331,3947.311264,1451.053257,935.670677,1705.401533,908.245599,5373.621980,...,31.422680,73.319588,1305.323556,361.610280,1306.115163,1358.837668,428.125935,22.994505,38.324176,88.632353
Pop % bacias,0.083869,0.001374,0.033301,0.052682,0.052872,0.019436,0.012533,0.022843,0.012165,0.071976,...,0.000421,0.000982,0.017484,0.004844,0.017495,0.018201,0.005734,0.000308,0.000513,0.001187
Pop % APS,0.082095,0.001345,0.032597,0.051568,0.051754,0.019025,0.012268,0.022360,0.011908,0.070454,...,0.000412,0.000961,0.017114,0.004741,0.017125,0.017816,0.005613,0.000301,0.000502,0.001162
Extensão de Rede (m),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Área (km²),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [282]:
resultado_final.to_excel(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Ijuí\inputpredim_ec.xlsx')